# 03 · Pydantic 模型如何变成 LangChain 的结构化输出

前两节分别讲了 [Pydantic 格式的使用](01-Pydantic格式的使用.ipynb) 和 [Pydantic 的高级特性](02-Pydantic的高级特性.ipynb)。这一节沿着一条数据的旅行，回答：

> `Person` 类如何告诉大模型要什么？模型返回了什么？最后是谁创建了 `Person(...)` 对象？

**大模型生成 JSON 内容或工具调用参数；本地的 SDK／LangChain 解析器调用 Pydantic，完成校验并创建 Python 对象。** 网络上传输的是可序列化的数据，不是 Python 类、函数或内存中的对象。

阅读顺序：① 总图 → ② 定义结构 → ③ 绑定结构 → ④ 两条调用路径 → ⑤ 本地校验 → ⑥ 真实调用 → ⑦ 排错。

本节沿用项目的 **Pydantic v2、LangChain 1.x、ChatOpenAI**，以**非流式 Chat Completions** 为例。源码细节核对版本：`langchain-openai 1.6.4`、`langchain-core 1.6.4`、`openai 3.19.0`、`pydantic 2.13.5`。其他厂商、Responses API、流式调用的包装细节可能不同。

**运行方式：** 选择本项目 `.venv` 内核，从上到下运行。默认全部为本地演示，不需要 API Key；第 6 节将 `RUN_LIVE` 改为 `True` 后，才会加载 `conf/.env` 并请求真实模型。模拟消息均明确标注，不代表服务端实测结果。

## 1. 先看全流程：发出结构要求，再把返回数据变成对象

![Pydantic 与 LangChain 结构化输出完整流程](assets/03-pydantic-structured-output-flow.svg)

上图分开了**构建链**和**执行链**：

```text
构建阶段（本地，不请求 LLM）
  Person 类 → with_structured_output(Person) → 得到可调用的 Runnable

执行阶段（invoke 时）
  输入文本 + Person 对应的 JSON Schema
      → 服务端 / LLM 生成结构化数据
      → 本地 SDK / LangChain 解析
      → Pydantic 校验并创建 Person 实例
      → 业务代码读取 person.name、person.age
```

`Person` 在这里承担两份工作：**发出请求时提供结构说明；收到响应时提供校验和构造规则。**

| 参与者 | 负责什么 | 本节例子 |
|---|---|---|
| 你的业务代码 | 定义数据结构、输入任务、消费结果 | `Person`、提取人物信息、`person.age` |
| LangChain | 绑定结构化输出参数，串联模型和解析组件 | `with_structured_output`、`invoke` |
| 模型提供商 SDK | 序列化请求、处理响应；某些路径还会调用 Pydantic | OpenAI Python SDK |
| 服务端与 LLM | 根据输入和结构要求生成 JSON 内容或工具调用参数 | `"age": 30` |
| Pydantic | 在运行程序的 Python 进程中校验、转换数据、创建对象 | `Person.model_validate_json(...)` |

这里的“本地”指**运行 Python 程序的一侧**；程序部署在服务器时，对象就在那台服务器上创建。

## 2. 定义结构：Pydantic 类不是大语言模型

这里有两个容易混淆的“模型”：

- `Person(BaseModel)` 是**数据模型**：描述数据应该长什么样。
- `ChatOpenAI(...)` 是**聊天模型客户端**：负责调用远端 LLM。

把 `Person` **类本身**传给 `with_structured_output`，不需要提前创建一个 `Person(...)` 实例。

In [1]:
import json
from pydantic import BaseModel, ConfigDict, Field, ValidationError

class Person(BaseModel):
    """从原文中提取的人物信息。"""
    model_config = ConfigDict(extra="forbid")

    name: str = Field(description="姓名")
    age: int | None = Field(description="年龄；原文未提到时返回 null，不要猜测")
    occupation: str = Field(description="职业")

source_text = "张三是一名 30 岁的软件工程师"
example_data = {"name": "张三", "age": 30, "occupation": "软件工程师"}

print("传入的结构类：", Person.__name__)
print("原始输入：", source_text)

传入的结构类： Person
原始输入： 张三是一名 30 岁的软件工程师


### 2.1 类怎样变成服务端能理解的结构说明？

`Person.model_json_schema()` 返回一个 **dict 形式的 JSON Schema**。它描述字段，不包含这一条人物记录的数据。

`Field(description=...)` 帮助模型理解字段含义，但描述本身不是可执行校验规则；例如“不要猜测”不能保证模型一定不猜测。`extra="forbid"` 则会在本地拒绝多余字段，并体现在 Schema 的 `additionalProperties: false` 中。

SDK／LangChain 会根据具体 API 再转换、整理 Schema。**直接生成的 Pydantic Schema 不一定等于最终的 HTTP 请求体**，例如 OpenAI 严格输出格式还需要名称、`strict` 标志等包装。

In [2]:
person_schema = Person.model_json_schema()
print(json.dumps(person_schema, ensure_ascii=False, indent=2))
print("必填字段：", person_schema["required"])
print("age 的类型约束：", person_schema["properties"]["age"]["anyOf"])

{
  "additionalProperties": false,
  "description": "从原文中提取的人物信息。",
  "properties": {
    "name": {
      "description": "姓名",
      "title": "Name",
      "type": "string"
    },
    "age": {
      "anyOf": [
        {
          "type": "integer"
        },
        {
          "type": "null"
        }
      ],
      "description": "年龄；原文未提到时返回 null，不要猜测",
      "title": "Age"
    },
    "occupation": {
      "description": "职业",
      "title": "Occupation",
      "type": "string"
    }
  },
  "required": [
    "name",
    "age",
    "occupation"
  ],
  "title": "Person",
  "type": "object"
}
必填字段： ['name', 'age', 'occupation']
age 的类型约束： [{'type': 'integer'}, {'type': 'null'}]


本例的 `age` **允许为 `None`，但仍是必填字段**：它没有默认值。

```json
{"name": "张三", "age": null, "occupation": "医生"}
```

上面的数据合法；如果直接省略 `age`，则不满足这个 Pydantic 类的要求。`int | None`／`Optional[int]` 表示“值可为空”，不是“字段可省略”。

本节用必填且可空的字段表达未知年龄，不给人物信息编造默认年龄。

## 3. 绑定结构：with_structured_output 只是在组装一条链

先创建一个**只用于本地观察的客户端**，使用占位 Key 和保留的 `.invalid` 域名。不要对它调用 `invoke`；第 6 节另建真实客户端。

调用 `with_structured_output` 不会向大模型发请求，也不会训练模型或生成一份人物信息。它返回一个新的 **Runnable**（可调用组件），在本例中是 `RunnableSequence`：

```text
绑定了输出参数的模型 → 对应的解析器
```

默认 `include_raw=False` 时，成功调用这条链得到 `Person` 实例。原始 `llm.invoke(...)` 通常返回的则是 `AIMessage`。

In [3]:
from langchain_openai import ChatOpenAI

inspection_llm = ChatOpenAI(
    model="gpt-6-luna",  # 延续前两节的名称；构建链不会检查远端是否支持
    api_key="offline-placeholder",
    base_url="https://offline.invalid/v1",
    use_responses_api=False,  # 固定为本节讲解的 Chat Completions 路径
    max_retries=0,
)

json_chain = inspection_llm.with_structured_output(
    Person, method="json_schema", strict=True,
)
tool_chain = inspection_llm.with_structured_output(
    Person, method="function_calling", strict=False,
)

print("json_schema 返回：", type(json_chain).__name__)
print("function_calling 返回：", type(tool_chain).__name__)
print("json_schema 的绑定参数名：", list(json_chain.first.kwargs))
print("绑定的 response_format 仍是 Person 类：",
      json_chain.first.kwargs["response_format"] is Person)
print("工具路径最后一步：", type(tool_chain.last).__name__)

json_schema 返回： RunnableSequence
function_calling 返回： RunnableSequence
json_schema 的绑定参数名： ['response_format', 'ls_structured_output_format']
绑定的 response_format 仍是 Person 类： True
工具路径最后一步： PydanticToolsParser


这里查看 `.first`、`.last`、绑定参数，是针对当前版本的学习观察；业务代码只需要使用 `invoke` 等公开调用接口。

**为什么绑定参数里仍然是 `Person` 类？** 因为这时还在本地。真正发送 HTTP 请求前，SDK 才把它转换成可序列化的 JSON Schema。

| 方式 | 结构要求放在哪里（Chat Completions） | 返回数据的主要位置 | 本地处理方式 |
|---|---|---|---|
| `json_schema` | `response_format` 内的 JSON Schema | 响应消息的 JSON 内容 | 本例由 SDK 调 Pydantic，LangChain 提取对象 |
| `function_calling` | `tools[].function.parameters`，并指定工具选择 | 工具调用的 `function.arguments` | LangChain 解析参数，再用 Pydantic 构造对象 |
| `json_mode` | `response_format={"type": "json_object"}`；字段结构要写入提示词 | 响应消息的 JSON 内容 | `PydanticOutputParser` 解析并校验 |

`json_mode` 要求合法 JSON，不等于按指定字段输出。本节重点拆解前两条路径，**显式指定 `method`**，避免把某个集成的默认值当成所有模型的共同规则。[LangChain 官方说明](https://docs.langchain.com/oss/python/langchain/models#structured-output)

## 4. 执行结构化输出：两条路径在哪里创建对象？

### 4.1 路径 A：json_schema → SDK 解析 → Person

以本节固定的 `ChatOpenAI + Chat Completions + Pydantic v2` 非流式路径为例：

```text
你的代码              LangChain                 OpenAI SDK                 服务端 / LLM
   │                      │                         │                           │
   ├─ invoke(文本) ──────>│                         │                           │
   │                      ├─ messages + Person ────>│                           │
   │                      │                         ├─ messages + JSON Schema ─>│
   │                      │                         │<─ content: JSON 字符串 ───┤
   │                      │                         │                           │
   │                      │                         ├─ Person.model_validate_json
   │                      │                         │  校验，创建 Person 实例
   │                      │<─ SDK 响应 + parsed ────┤
   │                      ├─ 包装成 AIMessage       │
   │                      │  additional_kwargs["parsed"] 保存对象
   │                      ├─ 最后的解析组件取出对象 │
   │<─ Person 实例 ────────┤                         │
```

**最后的解析组件可能只是“取对象”，因为 SDK 已经创建好了对象。** 这与“每条路径最后都先 `json.loads` 再构造对象”的说法不同。

下面手工提供一段**模拟的响应 JSON**，只拆解本地阶段，不请求模型。`model_validate_json` 一步完成 JSON 解析、Pydantic 校验和实例创建，无需再手动 `json.loads`。

In [4]:
from langchain_core.messages import AIMessage

# 模拟 LLM 返回的 content，不是真实模型调用结果。
simulated_content = json.dumps(example_data, ensure_ascii=False)

# 对应 SDK 内部所做的核心转换（这里手工演示）。
sdk_person = Person.model_validate_json(simulated_content)

# 对应 LangChain 对响应的包装（这里手工构造）。
simulated_json_message = AIMessage(
    content=simulated_content,
    additional_kwargs={"parsed": sdk_person},
)

# 调用当前 json_schema 链的最后一个组件；不会触发模型调用。
json_person = json_chain.last.invoke(simulated_json_message)

print("content 的类型：", type(simulated_json_message.content).__name__)
print("SDK 创建的对象：", repr(sdk_person))
print("LangChain 返回的对象：", repr(json_person))
print("最后一步取出同一个对象：", json_person is sdk_person)

content 的类型： str
SDK 创建的对象： Person(name='张三', age=30, occupation='软件工程师')
LangChain 返回的对象： Person(name='张三', age=30, occupation='软件工程师')
最后一步取出同一个对象： True


### 4.2 路径 B：function_calling → 工具参数 → Person

这条路径把 `Person` 转成名为 `Person` 的工具 Schema，让模型生成“调用这个工具需要的参数”：

```text
Person 类 → 工具 Schema（name + description + parameters）
                           │
输入文本 + tools + tool_choice → 服务端 / LLM
                           │
                           ↓
工具调用：name="Person", arguments='{"name":"张三","age":30,...}'
                           │
                           ↓ 本地解析 JSON 参数
AIMessage.tool_calls[0]["args"] = {"name": "张三", "age": 30, ...}
                           │
                           ↓ PydanticToolsParser
                       Person(**args)
                           │
                           ↓
                      Person 实例
```

**这里不需要真的执行一个远端工具，也没有函数体供 LLM 执行。** 我们只是借工具调用协议得到结构化参数，随后在本地构造数据对象；通常无需再把工具结果发回模型进行第二轮对话。

In [5]:
from langchain_core.utils.function_calling import convert_to_openai_tool

person_tool = convert_to_openai_tool(Person, strict=False)
print("工具名称：", person_tool["function"]["name"])
print("工具参数：", json.dumps(
    person_tool["function"]["parameters"], ensure_ascii=False, indent=2,
))
print("指定模型选择的工具：", tool_chain.first.kwargs["tool_choice"])

工具名称： Person
工具参数： {
  "additionalProperties": false,
  "properties": {
    "name": {
      "description": "姓名",
      "type": "string"
    },
    "age": {
      "anyOf": [
        {
          "type": "integer"
        },
        {
          "type": "null"
        }
      ],
      "description": "年龄；原文未提到时返回 null，不要猜测"
    },
    "occupation": {
      "description": "职业",
      "type": "string"
    }
  },
  "required": [
    "name",
    "age",
    "occupation"
  ],
  "type": "object"
}
指定模型选择的工具： {'type': 'function', 'function': {'name': 'Person'}}


下面同样使用**模拟消息**。服务端协议中的 `function.arguments` 是 JSON 字符串；在 LangChain 的标准化 `AIMessage.tool_calls` 中，`args` 已经是 Python 字典。`raw.content` 可以为空，结构化数据在工具调用里。

In [6]:
from langchain_core.output_parsers.openai_tools import PydanticToolsParser

simulated_tool_message = AIMessage(
    content="",
    tool_calls=[{
        "name": "Person",
        "args": example_data,
        "id": "call_demo_01",
        "type": "tool_call",
    }],
)

# with_structured_output(Person, method="function_calling") 使用的解析器。
tool_parser = PydanticToolsParser(tools=[Person], first_tool_only=True)
tool_person = tool_parser.invoke(simulated_tool_message)

print("content：", repr(simulated_tool_message.content))
print("工具参数：", simulated_tool_message.tool_calls[0]["args"])
print("返回结果：", repr(tool_person))
print("返回类型：", type(tool_person).__name__)
print("与 JSON 路径的数据一致：", tool_person == json_person)

content： ''
工具参数： {'name': '张三', 'age': 30, 'occupation': '软件工程师'}
返回结果： Person(name='张三', age=30, occupation='软件工程师')
返回类型： Person
与 JSON 路径的数据一致： True


### 4.3 把“传输了什么”和“创建了什么”区分开

| 阶段 | 示例 | Python 侧类型 / 含义 |
|---|---|---|
| 结构定义 | `Person` | 类；创建对象的规则 |
| 请求的结构说明 | `Person.model_json_schema()` 经 API 适配 | `dict`，序列化后随请求发送 |
| 响应的 JSON 内容 / 工具参数文本 | `'{"name": "张三", "age": 30, ...}'` | `str`；此时不是 `Person` |
| 解析出的普通数据（工具路径可见） | `{"name": "张三", "age": 30, ...}` | `dict`；可用键读取 |
| 校验后的业务对象 | `Person(name="张三", age=30, ...)` | `Person` 实例；可用属性读取 |

上表的 JSON 内容是示意，实际有效 JSON 不能含 `...`。`AIMessage` 是 LangChain 的消息容器，`Person` 是业务数据对象，二者用途不同。`include_raw=True` 的 `raw` 也是已经包装过的消息，**不是原始 HTTP 字节**。

如果把 `Person.model_json_schema()` 这个**字典**传给 `with_structured_output`，本例集成返回的是普通字典；想直接得到 `Person` 实例，应传 `Person` 类。需要时也可以自己对字典调用 `Person.model_validate(data)`。

## 5. 收到数据后，Pydantic 在本地做什么？

对于本节的模型，可以用下面的概念流程理解校验；复杂 `before`／`wrap` 校验器会影响具体顺序，并非所有模型都固定执行完全相同的步骤。

```text
JSON / dict
   → 读取字段；检查缺失与多余字段；按定义处理默认值
   → 类型检查、允许的类型转换；递归处理嵌套模型
   → 字段约束和自定义校验规则
   → 成功：得到实例       失败：抛出 ValidationError
```

`json.loads(...)` 只解析 JSON 语法，不知道 `age` 应该是什么类型。Pydantic 负责这些业务结构规则。

三个入口要区分：`Person(**data)` 使用构造函数；`Person.model_validate(data)` 接收 Python 数据；`Person.model_validate_json(text)` 接收 JSON 文本。本节简单数据在这些入口得到相同结果，但严格模式与某些日期等类型可能有差异。[Pydantic 模型与校验入口](https://docs.pydantic.dev/latest/concepts/models/)

In [7]:
# JSON 语法正确，不代表数据符合 Person。
bad_data = {**example_data, "age": "三十岁"}
bad_json = json.dumps(bad_data, ensure_ascii=False)

print("json.loads 可以成功：", json.loads(bad_json))
try:
    Person.model_validate_json(bad_json)
except ValidationError as exc:
    error = exc.errors()[0]
    print("Pydantic 拒绝：", error["loc"], error["msg"])

# Pydantic 默认允许部分类型转换，例如数字字符串 -> int。
coerced = Person.model_validate({**example_data, "age": "30"})
print("默认模式：", repr(coerced), "| age 类型：", type(coerced.age).__name__)

# Pydantic 的 strict=True 控制本地类型转换。
try:
    Person.model_validate({**example_data, "age": "30"}, strict=True)
except ValidationError as exc:
    print("本地严格模式拒绝数字字符串：", exc.errors()[0]["msg"])

json.loads 可以成功： {'name': '张三', 'age': '三十岁', 'occupation': '软件工程师'}
Pydantic 拒绝： ('age',) Input should be a valid integer, unable to parse string as an integer
默认模式： Person(name='张三', age=30, occupation='软件工程师') | age 类型： int
本地严格模式拒绝数字字符串： Input should be a valid integer


### 5.1 两种 strict，作用在不同位置

| 写法 | 在哪里生效 | 约束什么 |
|---|---|---|
| `with_structured_output(..., strict=True)` | 请求参数 / 支持该能力的服务端 | 按服务端支持的 JSON Schema 严格生成 |
| `ConfigDict(strict=True)` 或 `model_validate(..., strict=True)` | Python 进程内的 Pydantic | 本地校验时是否允许类型转换 |

它们不是同一个开关。服务端严格输出依赖具体模型、接口和受支持的 Schema；拒绝、截断或不兼容的服务端仍可能导致调用失败。成功通过结构校验也不保证提取内容符合原文。

例如原文是“30 岁”，模型返回 `age=31`，类型仍然合法。要验证事实，需要额外比对原文或设计核验流程。[Pydantic 严格模式](https://docs.pydantic.dev/latest/concepts/strict_mode/)

### 5.2 缺字段、null 和默认值，是三种不同情况

第二节见到的默认值问题，发生在这里：**只有传入数据缺少字段时，本地默认值才有机会补上**；显式传入 `None` 不会自动替换成默认值。

下面仅为观察 Pydantic 默认值机制而使用 `32`，不把它用于本节真实的人物提取。

In [8]:
class DefaultAgeDemo(BaseModel):
    age: int | None = 32

print("缺字段：", DefaultAgeDemo.model_validate({}))
print("显式 null：", DefaultAgeDemo.model_validate_json('{"age": null}'))
print("明确提供年龄：", DefaultAgeDemo.model_validate({"age": 30}))

try:
    Person.model_validate({"name": "张三", "occupation": "医生"})
except ValidationError as exc:
    print("本节 Person 缺少 age：", exc.errors()[0]["msg"])

print("本节 Person 显式给空值：", Person.model_validate({
    "name": "张三", "age": None, "occupation": "医生",
}))

缺字段： age=32
显式 null： age=None
明确提供年龄： age=30
本节 Person 缺少 age： Field required
本节 Person 显式给空值： name='张三' age=None occupation='医生'


**自定义校验器也在本地运行。** `@field_validator`／`@model_validator` 的 Python 函数体不会被上传给模型。部分结构约束可以导出为 Schema，但任意 Python 业务规则不会自动变成服务端约束。下面用自定义规则禁止负年龄，观察解析器会怎样处理它。

In [9]:
from pydantic import field_validator

class CheckedPerson(Person):
    @field_validator("age")
    @classmethod
    def reject_negative_age(cls, age: int | None) -> int | None:
        if age is not None and age < 0:
            raise ValueError("年龄不能小于 0")
        return age

# 自定义 Python 校验器没有自动变成 age 的 JSON Schema 约束。
print("age 的 Schema 是否相同：",
      CheckedPerson.model_json_schema()["properties"]["age"]
      == Person.model_json_schema()["properties"]["age"])
try:
    CheckedPerson.model_validate({**example_data, "age": -1})
except ValidationError as exc:
    print("本地业务规则拦截：", exc.errors()[0]["msg"])

age 的 Schema 是否相同： True
本地业务规则拦截： Value error, 年龄不能小于 0


### 5.3 成功之后：业务代码拿到的是实例

读取属性、转回字典和序列化为 JSON 都在本地完成，不需要再次请求模型。注意 `model_json_schema()` 输出**结构说明**，`model_dump_json()` 输出**这条记录的数据**。

In [10]:
person = tool_person
print("对象类型：", type(person).__name__)
print("用属性访问：", person.name, person.age, person.occupation)
print("转为字典：", person.model_dump())
print("转为 JSON：", person.model_dump_json())

对象类型： Person
用属性访问： 张三 30 软件工程师
转为字典： {'name': '张三', 'age': 30, 'occupation': '软件工程师'}
转为 JSON： {"name":"张三","age":30,"occupation":"软件工程师"}


## 6. 把各步骤交回 LangChain：完整真实调用

理解拆解后，实际使用仍只需：**定义类 → 初始化客户端 → `with_structured_output` → `invoke` → 使用对象**。

下面使用项目的 `conf/.env`，识别现有的 `OPENAI_API_KEY`、`OPENAI_API_BASE`，也兼容 `OPENAI_BASE_URL`。从仓库根目录或本章目录启动 Notebook 都可以。已有环境变量优先，不打印 Key。

`RUN_LIVE=False` 时直接跳过；设为 `True` 后，使用 `OPENAI_MODEL`（默认延续前两节的 `gpt-6-luna`）调用**一次**模型。模型名称只是项目配置示例，接口必须实际支持所选 `METHOD`；兼容接口不代表所有结构化输出能力都兼容。

本节默认 `METHOD="json_schema"`。若你的接口只支持工具调用，可以手动改为 `"function_calling"`；代码不会自动切换或增加重试来掩盖问题。

In [11]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate

RUN_LIVE = False
METHOD = "json_schema"  # 也可改为 "function_calling"

if RUN_LIVE:
    project_root = next(
        (p for p in (Path.cwd(), *Path.cwd().parents)
         if (p / "pyproject.toml").is_file()),
        None,
    )
    if project_root is None:
        raise RuntimeError("请从仓库根目录或本章目录启动 Notebook。")
    load_dotenv(project_root / "conf" / ".env", override=False)
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("请在环境变量或 conf/.env 中设置 OPENAI_API_KEY。")
    if METHOD not in {"json_schema", "function_calling"}:
        raise ValueError("本示例只比较 json_schema 和 function_calling。")

    live_llm = ChatOpenAI(
        model=os.getenv("OPENAI_MODEL", "gpt-6-luna"),
        api_key=os.environ["OPENAI_API_KEY"],
        base_url=os.getenv("OPENAI_API_BASE") or os.getenv("OPENAI_BASE_URL"),
        use_responses_api=False,
        timeout=60,
        max_retries=0,
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", "从原文提取人物信息。不要猜测年龄，未提到时 age 返回 null。"),
        ("human", "{text}"),
    ])
    structured_llm = live_llm.with_structured_output(
        Person,
        method=METHOD,
        strict=True if METHOD == "json_schema" else False,
        include_raw=True,
    )
    live_chain = prompt | structured_llm
    output = live_chain.invoke({"text": source_text})

    print("raw 类型：", type(output["raw"]).__name__)
    print("raw.content：", repr(output["raw"].content))
    print("raw.tool_calls：", output["raw"].tool_calls)
    print("parsed：", repr(output["parsed"]))
    print("parsing_error：", output["parsing_error"])

    if output["parsing_error"] is not None:
        print("解析失败，请结合 raw 检查输出。")
    elif isinstance(output["parsed"], Person):
        live_person = output["parsed"]
        print("属性访问：", live_person.name, live_person.age)
        print("业务数据：", live_person.model_dump())
    else:
        print("没有得到 Person 对象，请检查原始消息与接口能力。")
else:
    print("已跳过真实模型调用；将 RUN_LIVE 改为 True 后运行本单元。")

已跳过真实模型调用；将 RUN_LIVE 改为 True 后运行本单元。


`include_raw=True` 时，成功返回值的形状如下（**示意，不是真实调用输出**）：

```python
{
    "raw": AIMessage(...),               # 消息容器，保留内容和元数据
    "parsed": Person(name="张三", age=30, occupation="软件工程师"),
    "parsing_error": None,
}
```

业务代码从 `output["parsed"]` 取对象。改为 `include_raw=False` 后，链会直接返回 `Person`，代码就应写成 `person = live_chain.invoke(...)`，不再访问 `output["parsed"]`。

构建 `prompt | structured_llm` 也不发请求；只有调用 `invoke` 才执行链。普通单次提取通常只需要一轮模型请求，JSON 解析和 Pydantic 校验不需要另一次 LLM 调用。

## 7. 失败发生在哪里？include_raw 不是全局异常捕获器

`include_raw=True` 捕获的是 **LangChain 最后解析阶段**的异常；它不会把鉴权、网络、服务端错误等所有问题都变成 `parsing_error`。

| 失败位置 | 例子 | 本节路径中的表现 |
|---|---|---|
| 请求前 / 远端接口 | Key 无效、Schema 不受支持、网络超时 | 通常直接抛异常，尚未得到 `raw` 字典 |
| SDK 解析阶段 | `json_schema` 路径的 JSON 不合法、Pydantic 校验失败、输出截断 | 可以在模型调用阶段直接抛异常，`include_raw` 未必能接住 |
| LangChain 最后解析阶段 | 工具参数中的 `age="三十岁"` | `include_raw=False` 抛异常；`True` 通常得到 `parsed=None` 和 `parsing_error` |
| 缺少预期数据 | 模型没有返回应有的工具调用 | 工具解析器可能返回 `None`，不一定抛异常；还需检查结果类型 |
| 内容不符合原文 | 输入 30 岁，输出 31 岁 | 类型合法，Pydantic 不会自动识别事实错误 |

例如服务端返回拒绝消息，当前 `json_schema` 最后的解析组件也可能抛出拒绝异常，随后被 `include_raw=True` 捕获。具体错误落点应结合原始消息和当前 SDK 版本判断。

下面用 LangChain 的 Runnable **复现工具路径的解析分支**：使用模拟消息，保留相同的解析器和 fallback 结构，不调用服务端。

In [12]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

parse_with_raw = RunnablePassthrough.assign(
    parsed=RunnableLambda(itemgetter("raw")) | tool_parser,
    parsing_error=lambda _: None,
).with_fallbacks(
    [RunnablePassthrough.assign(parsed=lambda _: None)],
    exception_key="parsing_error",
)

invalid_tool_message = AIMessage(
    content="",
    tool_calls=[{
        "name": "Person",
        "args": {**example_data, "age": "三十岁"},
        "id": "call_demo_invalid",
        "type": "tool_call",
    }],
)

for label, message in [
    ("成功", simulated_tool_message),
    ("校验失败", invalid_tool_message),
    ("未返回工具调用", AIMessage(content="这是一段普通回答")),
]:
    demo_output = parse_with_raw.invoke({"raw": message})
    error = demo_output["parsing_error"]
    print(label,
          "| parsed =", repr(demo_output["parsed"]),
          "| parsing_error =", type(error).__name__ if error else None)

成功 | parsed = Person(name='张三', age=30, occupation='软件工程师') | parsing_error = None
校验失败 | parsed = None | parsing_error = ValidationError
未返回工具调用 | parsed = None | parsing_error = None


解析失败不会自动变成正确结果。是否重试、补充提示词或交给人工，属于应用层策略。`with_structured_output` 不能等同于 Agent 的 `ToolStrategy` 重试机制，也不会把所有不支持结构化输出的接口自动降级为“提示词 + JSON 解析”。

如果要确保拿到了有效业务对象，除了检查 `parsing_error`，还应检查 `isinstance(output["parsed"], Person)`。真实调用阶段的异常则在调用边界按应用需求单独处理。

## 8. 对照源码与资料继续学习

本节的两条路径根据当前 `.venv` 源码核对。以下定位表可以解释前面每一个箭头；这些是阅读入口，业务代码无需导入 SDK 的私有函数。

| 组件 | 源码文件 / 查找的符号 | 对应职责 |
|---|---|---|
| LangChain OpenAI 集成 | `langchain_openai/chat_models/base.py`：`BaseChatOpenAI.with_structured_output` | 根据 `method` 绑定参数并选择解析器；组装 `include_raw` 分支 |
| LangChain OpenAI 集成 | 同文件：`_generate`、`_create_chat_result` | 调用 SDK，并把响应转换成 `AIMessage` |
| OpenAI SDK | `openai/lib/_parsing/_completions.py`：`type_to_response_format_param`、`_parse_content` | 将类适配为请求 Schema；解析返回内容 |
| OpenAI SDK | `openai/_compat.py`：`model_parse_json` | Pydantic v2 下调用 `model_validate_json` |
| LangChain OpenAI 集成 | `langchain_openai/chat_models/base.py`：`_oai_structured_outputs_parser` | 取出 SDK 解析结果；若结果为字典则用 Schema 类构造对象 |
| LangChain Core | `langchain_core/output_parsers/openai_tools.py`：`PydanticToolsParser.parse_result` | 查找目标类，使用工具参数构造 Pydantic 对象 |

官方资料：

- [LangChain：模型的结构化输出](https://docs.langchain.com/oss/python/langchain/models#structured-output)
- [Pydantic：模型、校验与序列化](https://docs.pydantic.dev/latest/concepts/models/)
- [Pydantic：自定义校验器](https://docs.pydantic.dev/latest/concepts/validators/)
- [Pydantic：严格模式](https://docs.pydantic.dev/latest/concepts/strict_mode/)

检查自己是否理解了流程：

1. `with_structured_output(Person)` 会立刻请求大模型吗？——不会；它组装 Runnable。
2. 网络上会发送 `Person` 的 Python 函数体吗？——不会；发送的是 API 参数中的 Schema 等数据。
3. 远端返回 `Person(name=...)` 这个 Python 对象吗？——不会；对象在 Python 程序运行的一侧创建。
4. 谁调用 Pydantic？——本节 `json_schema` 路径主要由 SDK 调用，工具路径由 LangChain 解析器调用。
5. 返回对象就一定符合事实吗？——只说明通过了当前结构与本地校验，事实还需要核验。